#### Ouput Parsers

First understand the problem 
- suppose if we ask an llm 
```
Explain the following information.
Name: jhon
Age: 25
city: benagaluru
```

- the model might return 
```
Name: John
Age: 25
City: Bangalore
```
this looks good 

- But tommorrow it may return 
``` John is 25 years old and lives in Bangalore.```
or
```
{
  "name": "John",
  "age": 25,
  "city": "Bangalore"
}
```
or 
``` The person's name is John. ```

- Notice the problem? : the ouput format is not Guranteed.
- for humans ``` no issues ```
- for programs ``` big issues ``` 
becuase python code expects a consistent format.


##### Overview

- Output Parsers in LangChain convert raw LLM responses into structured formats such as strings, JSON objects, lists, or validated Pydantic models. They help make AI outputs reliable and machine-readable for downstream applications like agents, RAG systems, OCR pipelines, and database integrations.


##### Why Output Parsers Exist

- Output parsers convert LLM output into a predictable structure.

- think 
```
Raw LLM Output
       ↓
Output Parser
       ↓
Structured Data
```
- example : 
```
LLM:
Name: John
Age: 25
```
- paraser 
```
{
  "name": "John",
  "age": 25
}
```

Now this any application will use it safely.


#### Overview
An Output Parser in LangChain is a core component used to structure, format, and parse raw text responses generated by Large Language Models (LLMs) into reliable, downstream-ready data formats like JSON, lists, Python objects, or Pydantic schemas. While modern models often support native structured outputs, output parsers remain crucial for streaming data, enforcing runtime validation, and processing output from models lacking built-in schema compliance.

#### Core Functions of Output Parsers 

1. Format Instructions : A method (get_format_instructions()) injected into the prompt template telling the LLM exactly how to structure its text response.
2. Parse Logic : A method (parse()) that takes the raw string output from the LLM and converts it into the desired Python data structure

##### Types of Output Parsers 
1. StrOutputParser
2. JsonOutputParser
3. PydanticOutputParser
4. CommaSeparatedListOutputParser
5. DatetimeOutputParser
6. Custom Output Parser

### 1.StrOutputParser

- Extract text content from model outputs as a string.
- Converts model outputs (such as AIMessage or AIMessageChunk objects) into plain text strings. It's the simplest output parser and is useful when you need string responses for downstream processing, display, or storage.
- Supports streaming, yielding text chunks as they're generated by the model.
- It is ideal for chat replies, text summarization, and building streaming pipelines.

**Why to Use this**
- Clean & Simple: Extracts the exact textual content from chat objects, ignoring unnecessary metadata like token usage or raw API structures.
- Streaming Support: Automatically yields text chunks as they are generated by the model.
- Pass-through: If the output is already a string, it is passed through unchanged






In [7]:
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()
model = ChatGroq(model = os.getenv("groq_model_name"))

parser = StrOutputParser()

# get string output from a model
message = model.invoke("Tell me a Joke")
result = parser.invoke(message)
print(result)

# with streaming - use transform() to process a stream.
stream = model.stream("Tell me a story")
for chunk in parser.transform(stream):
    print(chunk,end = "",flush=True)

A man walked into a library and asked the librarian, "Do you have any books on Pavlov's dogs and Schrödinger's cat?" 

The librarian replied, "It rings a bell, but I'm not sure if it's here or not."
Once upon a time, in a small village nestled in the rolling hills of a far-off land, there lived a young girl named Luna. Luna was a curious and adventurous soul, with a heart full of wonder and a mind full of questions.

As she grew up in the village, Luna was fascinated by the stories of her grandmother, a wise and kind woman who had lived in the village for many years. Her grandmother, Elara, was known for her vast knowledge of the natural world and the secrets it held. She would spend hours with Luna, teaching her about the stars, the phases of the moon, and the ancient magic that lay just beneath the surface of the world.

One day, as Luna was wandering through the village, she stumbled upon a hidden path she had never seen before. The path was overgrown with vines and shrubs, and it l

#### 2.JsonOutputParser
- The JsonOutputParser in LangChain converts raw text from a Large Language Model (LLM) into a structured Python dictionary or JSON object.
- It is widely considered the most reliable parser for structured data extraction when you are not using native model function calling.
- Unlike older parsers, it natively supports streaming, meaning it can yield partial JSON objects chunk-by-chunk as the model generates them.
- When used in streaming mode, it will yield partial JSON objects containing all the keys that have been returned so far.
- In streaming, if diff is set to True, yields JSONPatch operations describing the difference between the previous and the current object.

##### Implementation Options
- You can implement JSON parsing in LangChain using two primary workflows:
1. JsonOutputParser with Pydantic: Provides structured schemas, auto-generates prompt instructions, and forces strict type validation.
2. JsonOutputParser without Pydantic: Returns an arbitrary JSON structure when you don't need strict validation.


#### JSON Output Parser with Pydantic 
This approach defines a structured data model using Pydantic, passes it to the parser to auto-inject formatting instructions into your prompt, and returns a verified object.

In [12]:
import pydantic
import os
from dotenv import load_dotenv
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from pydantic import BaseModel,Field
load_dotenv()
# define the designed structured data that we can use 
class Joke(BaseModel):
    setup : str = Field(description="the question or setup of the joke")
    punchline: str = Field(description="the answer or funny conclusion")

# initalize the llm and the parser 
model = ChatGroq(model=os.getenv("groq_model_name"))
parser = JsonOutputParser(pydantic_object = Joke)
# print(model)

prompt = PromptTemplate(
    template = "Answer the user query.\n{format_instructions}\n{query}",
    input_variables = {"query"},
    partial_variables = {"format_instructions": parser.get_format_instructions()},
)

# chain them together using langchain xpression langauge 
chain = prompt | model | parser

response = chain.invoke({"query": "tell me a joke about programming."})
print(response)
print(type(response))

{'setup': 'Why do programmers prefer dark mode?', 'punchline': 'Because light attracts bugs.'}
<class 'dict'>


#####  Using JsonOutputParser Without a Schema
if you do not want to lock yourself into a strict Pydantic model, you can instantiate JsonOutputParser empty. It will instruct the LLM to return valid JSON and parse the raw string directly into a Python dictionary

In [13]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
model = ChatGroq(model=os.getenv("groq_model_name"))
parser = JsonOutputParser()

prompt = PromptTemplate(
    template="Provide a list of three major countries and their capitals.\n{format_instructions}",
    input_variables=[],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

chain = prompt | model | parser

response = chain.invoke({})
print(response)
# Output: {'countries': [{'name': 'United States', 'capital': 'Washington, D.C.'}, ...]}


[{'country': 'United States', 'capital': 'Washington D.C.'}, {'country': 'China', 'capital': 'Beijing'}, {'country': 'India', 'capital': 'New Delhi'}]


#### 5.CommaSeparatedListOutputParser
- The CommaSeparatedListOutputParser is a built-in utility in LangChain designed to instruct a Large Language Model (LLM) to format its response as a comma-separated list and then parse that raw string output directly into a Python or JavaScript native list/array

#### How it works 
- 1. Formatting Instructions: It provides a pre-baked prompt snippet telling the LLM exactly how to format the data (e.g., "Your response should be a list of comma separated values, eg: foo, bar, baz").
- 2. Parsing: When the LLM responds, the parser strips extraneous spacing and splits the string at each comma to return a clean native collection.



In [16]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq

# 1. initalize 2. Get instructions 3. Setup chain # Lanchain Expression Language. (LCEL)
parser = CommaSeparatedListOutputParser()
prompt = PromptTemplate(
    template="List 5 {subject}.\n{format_instructions}",
    input_variables=["subject"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

chain = prompt | model | parser
reponse = chain.invoke({"subject":"ice cream flavors"})
print(response)


[{'country': 'United States', 'capital': 'Washington D.C.'}, {'country': 'China', 'capital': 'Beijing'}, {'country': 'India', 'capital': 'New Delhi'}]


#### 5. DateTimeOutputParser
- In LangChain, the DatetimeOutputParser is a built-in module used to force a Large Language Model (LLM) to return a date and time string, which it then automatically parses into a native Python datetime object. This eliminates manual string slicing or datetime.strptime() logic when handling chronological outputs.

#### Core Functionality
- The DatetimeOutputParser implements two primary methods required by all LangChain output parsers:
- 1. get_format_instructions(): Generates clear prompt instructions (complete with randomly generated examples) telling the LLM exactly what format to follow (e.g., ISO 8601).
- 2. parse(): Converts the raw string output returned by the LLM into a standard Python datetime object 

#

In [19]:
# implementation
from langchain_core.output_parsers import  DatetimeOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq

# 1. Initialize the parser and define the expected format
# Example format: 'YYYY-MM-DD HH:mm:ss' (Standard Python strftime directives)
date_parser = DatetimeOutputParser(format="%Y-%m-%d %H:%M:%S")

# 2. Get format instructions to inject into the prompt
format_instructions = date_parser.get_format_instructions()
print("Instructions sent to LLM:\n", format_instructions)

# 3. Create the prompt template with format instructions
template = """Answer the user's question.

{format_instructions}

Question: {question}
Answer:"""

prompt = PromptTemplate.from_template(
    template=template,
    partial_variables={"format_instructions": format_instructions},
)

# 4. Initialize the LLM
model

# 5. Chain the components together using LCEL
chain = prompt | model | date_parser

# 6. Invoke the chain
result = chain.invoke({"question": "When did Apollo 11 land on the moon?"})

print("\nParsed Result Type:", type(result))
print("Parsed Result Value:", result)


ImportError: cannot import name 'DatetimeOutputParser' from 'langchain_core.output_parsers' (c:\Users\subramani.v\AppData\Local\Programs\Python\Python310\lib\site-packages\langchain_core\output_parsers\__init__.py)

#### Custom Output Parsers
- Sometimes built-in parsers aren't enough. we can create our own output parsing

In [22]:
# example
from langchain_core.output_parsers import (
    BaseOutputParser
)

# custom parser
class UpperParser(
    BaseOutputParser
):

    def parse(self, text):
        return text.upper()

parser = UpperParser()

parser.parse("hello")

'HELLO'